In [3]:

!apt-get update -qq
!apt-get install -y tesseract-ocr tesseract-ocr-hin tesseract-ocr-kan


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (5.3.4-1build5).
tesseract-ocr-hin is already the newest version (1:4.1.0-2).
tesseract-ocr-kan is already the newest version (1:4.1.0-2).
0 upgraded, 0 newly installed, 0 to remove and 82 not upgraded.


In [4]:

%pip install -q pytesseract opencv-python-headless pillow pandas matplotlib scikit-learn tqdm jiwer


In [5]:

from google.colab import drive

drive.mount('/content/drive')


Mounted at /content/drive


In [6]:

from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/BharatDocAI")

directories = [
    "data/raw_documents",
    "data/processed",
    "data/annotations",
    "data/ocr_outputs",
    "data/splits",
    "notebooks",
    "src/data_generation",
    "src/ocr",
    "src/extraction",
    "src/evaluation",
    "experiments/baselines",
    "experiments/finetuning",
    "models/checkpoints",
    "outputs/metrics",
    "outputs/visualizations",
    "reports"
]

for directory in directories:
    (PROJECT_ROOT / directory).mkdir(
        parents=True,
        exist_ok=True
    )

print("Project directory created successfully!")
print(PROJECT_ROOT)


Project directory created successfully!
/content/drive/MyDrive/BharatDocAI


In [7]:

import sys
import torch
import platform
import subprocess
import pytesseract

print("=" * 50)
print("BHARATDOC AI - ENVIRONMENT AUDIT")
print("=" * 50)

print("\n1. Python")
print(sys.version)

print("\n2. Operating System")
print(platform.platform())

print("\n3. PyTorch")
print(torch.__version__)

print("\n4. GPU Availability")
print(torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)
else:
    print("No GPU assigned. CPU mode enabled.")

print("\n5. Tesseract Version")
print(pytesseract.get_tesseract_version())

print("\n6. Installed OCR Languages")
languages = pytesseract.get_languages(config="")

for language in languages:
    print("-", language)

print("\n7. Project Directory")
print(PROJECT_ROOT)
print("Exists:", PROJECT_ROOT.exists())

print("\n" + "=" * 50)
print("ENVIRONMENT AUDIT COMPLETED")
print("=" * 50)


BHARATDOC AI - ENVIRONMENT AUDIT

1. Python
3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]

2. Operating System
Linux-6.6.159+-x86_64-with-glibc2.39

3. PyTorch
2.14.1+cu130

4. GPU Availability
True
GPU: Tesla T4
CUDA: 13.0

5. Tesseract Version
5.3.4

6. Installed OCR Languages
- eng
- hin
- kan
- osd

7. Project Directory
/content/drive/MyDrive/BharatDocAI
Exists: True

ENVIRONMENT AUDIT COMPLETED


In [8]:

!apt-get install -y fonts-noto-core
!fc-cache -f -v


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  fonts-noto-mono
The following NEW packages will be installed:
  fonts-noto-core fonts-noto-mono
0 upgraded, 2 newly installed, 0 to remove and 82 not upgraded.
Need to get 13.7 MB of archives.
After this operation, 44.8 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble/main amd64 fonts-noto-mono all 20201225-2 [435 kB]
Get:2 http://archive.ubuntu.com/ubuntu noble/main amd64 fonts-noto-core all 20201225-2 [13.3 MB]
Fetched 13.7 MB in 1s (13.1 MB/s)
Selecting previously unselected package fonts-noto-mono.
(Reading database ... 127332 files and directories currently installed.)
Preparing to unpack .../fonts-noto-mono_20201225-2_all.deb ...
Unpacking fonts-noto-mono (20201225-2) ...
Selecting previously unselected package fonts-noto-core.
Preparing to unpack .../fonts-noto-core_20201225-2_all.deb ...
Unp

In [9]:

import subprocess

result = subprocess.run(
    ["fc-match", "Noto Sans Kannada"],
    capture_output=True,
    text=True
)

print("Kannada font:")
print(result.stdout)

result = subprocess.run(
    ["fc-match", "Noto Sans Devanagari"],
    capture_output=True,
    text=True
)

print("Hindi font:")
print(result.stdout)


Kannada font:
NotoSansKannada-Regular.ttf: "Noto Sans Kannada" "Regular"

Hindi font:
NotoSansDevanagari-Regular.ttf: "Noto Sans Devanagari" "Regular"



In [10]:

from PIL import Image, ImageDraw, ImageFont
from pathlib import Path

import json

DATA_DIR = PROJECT_ROOT / "data/raw_documents"
ANNOTATION_DIR = PROJECT_ROOT / "data/annotations"

samples = {
    "english": {
        "font": "/usr/share/fonts/truetype/noto/NotoSans-Regular.ttf",
        "language": "eng",
        "text": [
            "INVOICE",
            "Name: Ravi Kumar",
            "Date: 12/08/2026",
            "Invoice No: INV-1045",
            "Address: Bengaluru",
            "Amount: 2500"
        ]
    },
    "hindi": {
        "font": "/usr/share/fonts/truetype/noto/NotoSansDevanagari-Regular.ttf",
        "language": "hin",
        "text": [
            "चालान",
            "नाम: रवि कुमार",
            "दिनांक: 12/08/2026",
            "चालान संख्या: INV-1045",
            "पता: बेंगलुरु",
            "राशि: 2500"
        ]
    },
    "kannada": {
        "font": "/usr/share/fonts/truetype/noto/NotoSansKannada-Regular.ttf",
        "language": "kan",
        "text": [
            "ಸರಕುಪಟ್ಟಿ",
            "ಹೆಸರು: ರವಿ ಕುಮಾರ್",
            "ದಿನಾಂಕ: 12/08/2026",
            "ಸರಕುಪಟ್ಟಿ ಸಂಖ್ಯೆ: INV-1045",
            "ವಿಳಾಸ: ಬೆಂಗಳೂರು",
            "ಮೊತ್ತ: 2500"
        ]
    }
}

ground_truth = {}

for language, sample in samples.items():

    image = Image.new("RGB", (1400, 650), "white")
    draw = ImageDraw.Draw(image)

    font = ImageFont.truetype(sample["font"], 42)

    y = 60

    for line in sample["text"]:
        draw.text(
            (70, y),
            line,
            font=font,
            fill="black"
        )
        y += 85

    image_path = DATA_DIR / f"{language}_invoice.png"
    image.save(image_path)

    ground_truth[language] = {
        "language": sample["language"],
        "text": "\n".join(sample["text"])
    }

    print("Generated:", image_path)

with open(
    ANNOTATION_DIR / "phase0_ground_truth.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        ground_truth,
        f,
        ensure_ascii=False,
        indent=4
    )

print("\nThree sample documents generated.")


Generated: /content/drive/MyDrive/BharatDocAI/data/raw_documents/english_invoice.png
Generated: /content/drive/MyDrive/BharatDocAI/data/raw_documents/hindi_invoice.png
Generated: /content/drive/MyDrive/BharatDocAI/data/raw_documents/kannada_invoice.png

Three sample documents generated.


In [11]:

import pytesseract
import json

from PIL import Image
from pathlib import Path

results = {}

for language in ["english", "hindi", "kannada"]:

    image_path = DATA_DIR / f"{language}_invoice.png"

    image = Image.open(image_path)

    lang_code = samples[language]["language"]

    extracted_text = pytesseract.image_to_string(
        image,
        lang=lang_code,
        config="--psm 6"
    )

    results[language] = extracted_text

    print("\n" + "=" * 60)
    print(language.upper())
    print("=" * 60)
    print(extracted_text)

output_path = PROJECT_ROOT / "data/ocr_outputs/phase0_ocr.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

print("\nOCR results saved:", output_path)



ENGLISH
INVOICE

Name: Ravi Kumar
Date: 12/08/2026
Invoice No: INV-1045
Address: Bengaluru
Amount: 2500


HINDI
चालान

नाम: रवि कुमार

दिनांक: 42/08/2026
चालान संख्या: ॥00- 045
पता: बेंगलुरु

राशि: 2500


KANNADA
ಸರಕುಪಟ್ಟಿ

ಹೆಸರು: ರವಿ ಕುಮಾರ್‌
ದಿನಾಂಕ: 12/08/2026
ಸರಕುಪಟ್ಟಿ ಸಂಖ್ಯೆ: 111-1045
ವಿಳಾಸ: ಬೆಂಗಳೂರು

ಮೊತ್ತ: 2500


OCR results saved: /content/drive/MyDrive/BharatDocAI/data/ocr_outputs/phase0_ocr.json


In [12]:

import json
import pandas as pd
from jiwer import cer, wer

ground_truth_path = (
    PROJECT_ROOT / "data/annotations/phase0_ground_truth.json"
)

ocr_path = (
    PROJECT_ROOT / "data/ocr_outputs/phase0_ocr.json"
)

with open(ground_truth_path, encoding="utf-8") as f:
    ground_truth = json.load(f)

with open(ocr_path, encoding="utf-8") as f:
    predictions = json.load(f)


def normalize_text(text):
    return " ".join(text.split())


evaluation_results = []

for language in ["english", "hindi", "kannada"]:

    reference = normalize_text(
        ground_truth[language]["text"]
    )

    prediction = normalize_text(
        predictions[language]
    )

    character_error = cer(reference, prediction)
    word_error = wer(reference, prediction)

    evaluation_results.append({
        "Language": language,
        "CER": character_error,
        "WER": word_error,
        "Character Accuracy": max(0, 1-character_error)
    })

results_df = pd.DataFrame(evaluation_results)

print(results_df.to_string(index=False))

output_path = (
    PROJECT_ROOT / "outputs/metrics/phase0_ocr_metrics.csv"
)

results_df.to_csv(output_path, index=False)

print("\nMetrics saved:", output_path)


Language      CER      WER  Character Accuracy
 english 0.000000 0.000000            1.000000
   hindi 0.057471 0.230769            0.942529
 kannada 0.039604 0.153846            0.960396

Metrics saved: /content/drive/MyDrive/BharatDocAI/outputs/metrics/phase0_ocr_metrics.csv


In [13]:

%pip install -q easyocr


In [14]:

import torch
import easyocr

print("EasyOCR:", easyocr.__version__)
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


EasyOCR: 1.7.2
PyTorch: 2.14.1+cu130
GPU available: True
GPU: Tesla T4


In [15]:

import easyocr

use_gpu = torch.cuda.is_available()

print("Using GPU:", use_gpu)

reader_en = easyocr.Reader(
    ["en"],
    gpu=use_gpu
)

reader_hi = easyocr.Reader(
    ["hi", "en"],
    gpu=use_gpu
)

reader_kn = easyocr.Reader(
    ["kn", "en"],
    gpu=use_gpu
)

print("All EasyOCR readers initialized!")


Using GPU: True
Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% CompleteAll EasyOCR readers initialized!


In [16]:

from pathlib import Path
import json

image_paths = {
    "english": DATA_DIR / "english_invoice.png",
    "hindi": DATA_DIR / "hindi_invoice.png",
    "kannada": DATA_DIR / "kannada_invoice.png"
}

readers = {
    "english": reader_en,
    "hindi": reader_hi,
    "kannada": reader_kn
}

easyocr_results = {}

for language, reader in readers.items():

    print("\n" + "=" * 60)
    print(language.upper())
    print("=" * 60)

    results = reader.readtext(
        str(image_paths[language]),
        detail=1,
        paragraph=False
    )

    # Preserve detected text in reading order
    extracted_text = "\n".join(
        item[1] for item in results
    )

    easyocr_results[language] = extracted_text

    for bounding_box, text, confidence in results:
        print(f"{text} | Confidence: {confidence:.4f}")

    print("\nCombined text:")
    print(extracted_text)

output_path = (
    PROJECT_ROOT / "data/ocr_outputs/easyocr_results.json"
)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(
        easyocr_results,
        f,
        ensure_ascii=False,
        indent=4
    )

print("\nEasyOCR results saved:", output_path)



ENGLISH
INVOICE | Confidence: 0.9712
Name: Ravi Kumar | Confidence: 0.9781
Date: 12/08/2026 | Confidence: 0.8613
Invoice No: INV-1045 | Confidence: 0.9821
Address: Bengaluru | Confidence: 0.9998
Amount: 2500 | Confidence: 0.9058

Combined text:
INVOICE
Name: Ravi Kumar
Date: 12/08/2026
Invoice No: INV-1045
Address: Bengaluru
Amount: 2500

HINDI
चालान | Confidence: 0.6791
नामः रवि कुमार | Confidence: 0.7817
दिनांकः १२/०8/२०२६ | Confidence: 0.4076
चालान संख्याः `DD-1०४५ | Confidence: 0.3790
पताः | Confidence: 0.6091
राशिः २५०० | Confidence: 0.8195
बेंगलुरु | Confidence: 0.5701

Combined text:
चालान
नामः रवि कुमार
दिनांकः १२/०8/२०२६
चालान संख्याः `DD-1०४५
पताः
राशिः २५००
बेंगलुरु

KANNADA
ಸರಕುಪಟ್ಟಿ | Confidence: 0.9946
ಹೆಸರು: ರವಿ ಕುಮಾರ್ | Confidence: 0.9477
ದಿನಾಂಕ: 12 /08 | 2026 | Confidence: 0.7254
ಸರಕುಪಟ್ಟಿ ಸಂಖ್ಯೆ: [LL-1045 | Confidence: 0.4822
ವಿಳಾಸ: ಬೆಂಗಳೂರು | Confidence: 0.7721
ಮೊತ್ತ: 2500 | Confidence: 0.8381

Combined text:
ಸರಕುಪಟ್ಟಿ
ಹೆಸರು: ರವಿ ಕುಮಾರ್
ದಿನಾಂಕ: 12 /08 | 2026
ಸರಕುಪಟ್

In [17]:

import pandas as pd
import json
from jiwer import cer, wer

with open(
    PROJECT_ROOT / "data/annotations/phase0_ground_truth.json",
    encoding="utf-8"
) as f:
    ground_truth = json.load(f)

def normalize_text(text):
    return " ".join(text.split())

easyocr_metrics = []

for language in ["english", "hindi", "kannada"]:

    reference = normalize_text(
        ground_truth[language]["text"]
    )

    prediction = normalize_text(
        easyocr_results[language]
    )

    character_error = cer(reference, prediction)
    word_error = wer(reference, prediction)

    easyocr_metrics.append({
        "Language": language,
        "CER": character_error,
        "WER": word_error,
        "Character Accuracy": max(0, 1-character_error)
    })

easyocr_df = pd.DataFrame(easyocr_metrics)

print(easyocr_df.to_string(index=False))

easyocr_df.to_csv(
    PROJECT_ROOT / "outputs/metrics/easyocr_metrics.csv",
    index=False
)

print("\nEasyOCR evaluation saved!")


Language      CER      WER  Character Accuracy
 english 0.000000 0.000000            1.000000
   hindi 0.413793 0.692308            0.586207
 kannada 0.069307 0.384615            0.930693

EasyOCR evaluation saved!


In [18]:

import json
import unicodedata
import re
import pandas as pd
from jiwer import cer, wer

with open(
    PROJECT_ROOT / "data/annotations/phase0_ground_truth.json",
    encoding="utf-8"
) as f:
    ground_truth = json.load(f)

with open(
    PROJECT_ROOT / "data/ocr_outputs/easyocr_results.json",
    encoding="utf-8"
) as f:
    easyocr_predictions = json.load(f)

def normalize_ocr_text(text):

    # Convert Unicode decimal digits into ASCII digits
    normalized = ""

    for char in text:
        try:
            normalized += str(unicodedata.decimal(char))
        except (ValueError, TypeError):
            normalized += char

    # Normalize equivalent colon characters
    normalized = normalized.replace("ः", ":")
    normalized = normalized.replace("：", ":")

    # Normalize whitespace
    normalized = re.sub(r"\s+", " ", normalized)

    # Remove unnecessary spaces around punctuation
    normalized = re.sub(r"\s*([:/-])\s*", r"\1", normalized)
    normalized = normalized.strip()

    return normalized


comparison = []

for language in ["english", "hindi", "kannada"]:

    reference = ground_truth[language]["text"]
    prediction = easyocr_predictions[language]

    raw_cer = cer(reference, prediction)
    raw_wer = wer(reference, prediction)

    normalized_reference = normalize_ocr_text(reference)
    normalized_prediction = normalize_ocr_text(prediction)

    normalized_cer = cer(
        normalized_reference,
        normalized_prediction
    )

    normalized_wer = wer(
        normalized_reference,
        normalized_prediction
    )

    comparison.append({
        "Language": language,
        "Raw CER": raw_cer,
        "Normalized CER": normalized_cer,
        "Raw WER": raw_wer,
        "Normalized WER": normalized_wer
    })

comparison_df = pd.DataFrame(comparison)

print(comparison_df.to_string(index=False))

comparison_df.to_csv(
    PROJECT_ROOT / "outputs/metrics/easyocr_normalized_metrics.csv",
    index=False
)

print("\nNormalized metrics saved successfully!")


Language  Raw CER  Normalized CER  Raw WER  Normalized WER
 english 0.000000        0.000000    0.000           0.000
   hindi 0.425287        0.256098    0.875           0.375
 kannada 0.069307        0.062500    0.625           0.500

Normalized metrics saved successfully!


In [19]:

%pip install -q paddleocr paddlepaddle


In [20]:

import paddle
import paddleocr

print("PaddlePaddle:", paddle.__version__)
print("PaddleOCR:", paddleocr.__version__)
print("Paddle device:", paddle.device.get_device())

# Use CPU for the initial compatibility test.
# We will investigate Paddle GPU installation separately.
paddle.set_device("cpu")

print("Selected device:", paddle.device.get_device())
print("PaddleOCR installation successful!")


/usr/local/lib/python3.13/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


PaddlePaddle: 3.3.1
PaddleOCR: 3.7.0
Paddle device: cpu
Selected device: cpu
PaddleOCR installation successful!


In [21]:

import paddle
import paddleocr

print("PaddlePaddle:", paddle.__version__)
print("PaddleOCR:", paddleocr.__version__)
print("Paddle device:", paddle.device.get_device())

# Use CPU for the initial compatibility test.
# We will investigate Paddle GPU installation separately.
paddle.set_device("cpu")

print("Selected device:", paddle.device.get_device())
print("PaddleOCR installation successful!")


PaddlePaddle: 3.3.1
PaddleOCR: 3.7.0
Paddle device: cpu
Selected device: cpu
PaddleOCR installation successful!


In [22]:

from paddleocr import PaddleOCR

paddle_en = PaddleOCR(
    lang="en",
    device="cpu",
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False
)

print("PaddleOCR English initialized!")


Creating model: ('PP-OCRv6_medium_det', None, None)
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.
Using official model (PP-OCRv6_medium_det), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-OCRv6_medium_det`.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Creating model: ('PP-OCRv6_medium_rec', None, None)
Using official model (PP-OCRv6_medium_rec), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-OCRv6_medium_rec`.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

PaddleOCR English initialized!


In [23]:

paddle_hi = PaddleOCR(
    lang="hi",
    device="cpu",
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False
)

print("PaddleOCR Hindi initialized!")


Creating model: ('PP-OCRv5_server_det', None, None)
Using official model (PP-OCRv5_server_det), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-OCRv5_server_det`.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('devanagari_PP-OCRv5_mobile_rec', None, None)
Using official model (devanagari_PP-OCRv5_mobile_rec), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/devanagari_PP-OCRv5_mobile_rec`.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

PaddleOCR Hindi initialized!


In [26]:
import paddle
import paddleocr
import paddlex

print("PaddlePaddle:", paddle.__version__)
print("PaddleOCR:", paddleocr.__version__)
print("PaddleX:", paddlex.__version__)

PaddlePaddle: 3.3.1
PaddleOCR: 3.7.0
PaddleX: 3.7.2


In [27]:
from paddleocr import PaddleOCR

paddle_en = PaddleOCR(
    lang="en",
    device="cpu",
    enable_mkldnn=False,
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False
)

paddle_hi = PaddleOCR(
    lang="hi",
    device="cpu",
    enable_mkldnn=False,
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False
)

print("PaddleOCR readers initialized with oneDNN disabled!")

Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv6_medium_det`.
Creating model: ('PP-OCRv6_medium_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv6_medium_rec`.
Creating model: ('PP-OCRv5_server_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-OCRv5_server_det`.
Creating model: ('devanagari_PP-OCRv5_mobile_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/devanagari_PP-OCRv5_mobile_rec`.


PaddleOCR readers initialized with oneDNN disabled!


In [28]:
for language, reader in {
    "english": paddle_en,
    "hindi": paddle_hi
}.items():

    print("\n", "=" * 50)
    print(language.upper())
    print("=" * 50)

    results = reader.predict(str(image_paths[language]))

    for result in results:
        data = result.json

        if callable(data):
            data = data()

        if isinstance(data, str):
            data = json.loads(data)

        if "res" in data:
            data = data["res"]

        for text, score in zip(
            data.get("rec_texts", []),
            data.get("rec_scores", [])
        ):
            print(f"{text} | Confidence: {score:.4f}")


ENGLISH
INVOICE | Confidence: 1.0000
Name: Ravi Kumar | Confidence: 0.9988
Date: 12/08/2026 | Confidence: 0.9942
Invoice No: INV-1045 | Confidence: 0.9999
Address: Bengaluru | Confidence: 0.9935
Amount: 2500 | Confidence: 0.9994

HINDI
चालान | Confidence: 1.0000
नाम: रवि कुमार | Confidence: 0.9879
दिनांक: 12/08/2026 | Confidence: 0.9835
चालान संख्या: -1045 | Confidence: 0.9777
पता: बेंगलुर | Confidence: 0.9618
राशि: 2500 | Confidence: 0.9679


In [30]:
print("Available keys:", paddle_results.keys())
print("Current results:", paddle_results)

Available keys: dict_keys([])
Current results: {}


In [31]:
import json

paddle_results = {}

for language, reader in {
    "english": paddle_en,
    "hindi": paddle_hi
}.items():

    print(f"\nProcessing {language}...")

    results = reader.predict(str(image_paths[language]))

    extracted_lines = []

    for result in results:

        data = result.json

        if callable(data):
            data = data()

        if isinstance(data, str):
            data = json.loads(data)

        if "res" in data:
            data = data["res"]

        texts = data.get("rec_texts", [])

        extracted_lines.extend(texts)

    paddle_results[language] = "\n".join(extracted_lines)

    print(paddle_results[language])

# Save results
with open(
    PROJECT_ROOT / "data/ocr_outputs/paddleocr_results.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(paddle_results, f, ensure_ascii=False, indent=4)

print("\nPaddleOCR results saved successfully!")


Processing english...
INVOICE
Name: Ravi Kumar
Date: 12/08/2026
Invoice No: INV-1045
Address: Bengaluru
Amount: 2500

Processing hindi...
चालान
नाम: रवि कुमार
दिनांक: 12/08/2026
चालान संख्या: -1045
पता: बेंगलुर
राशि: 2500

PaddleOCR results saved successfully!


In [33]:

import json
import pandas as pd
from jiwer import cer, wer

# Load ground truth
with open(
    PROJECT_ROOT / "data/annotations/phase0_ground_truth.json",
    encoding="utf-8"
) as f:
    ground_truth = json.load(f)


def normalize_text(text):
    return " ".join(text.split())


paddle_metrics = []

for language in ["english", "hindi"]:

    reference = normalize_text(
        ground_truth[language]["text"]
    )

    prediction = normalize_text(
        paddle_results[language]
    )

    character_error = cer(reference, prediction)
    word_error = wer(reference, prediction)

    paddle_metrics.append({
        "Language": language,
        "CER": character_error,
        "WER": word_error,
        "Character Accuracy": max(0, 1 - character_error)
    })

paddle_df = pd.DataFrame(paddle_metrics)

print("=" * 65)
print("PADDLEOCR EVALUATION RESULTS")
print("=" * 65)

print(paddle_df.to_string(index=False))

# Save results separately
output_path = (
    PROJECT_ROOT / "outputs/metrics/paddleocr_metrics.csv"
)

paddle_df.to_csv(output_path, index=False)

print("\nMetrics saved:", output_path)


PADDLEOCR EVALUATION RESULTS
Language      CER      WER  Character Accuracy
 english 0.000000 0.000000            1.000000
   hindi 0.045977 0.153846            0.954023

Metrics saved: /content/drive/MyDrive/BharatDocAI/outputs/metrics/paddleocr_metrics.csv


In [34]:

import pandas as pd

# Load individual evaluation results
tesseract_df = pd.read_csv(
    PROJECT_ROOT / "outputs/metrics/phase0_ocr_metrics.csv"
)

easyocr_df = pd.read_csv(
    PROJECT_ROOT / "outputs/metrics/easyocr_metrics.csv"
)

paddle_df = pd.read_csv(
    PROJECT_ROOT / "outputs/metrics/paddleocr_metrics.csv"
)

# Identify each OCR engine
tesseract_df["OCR Engine"] = "Tesseract"
easyocr_df["OCR Engine"] = "EasyOCR"
paddle_df["OCR Engine"] = "PaddleOCR"

# Combine
comparison = pd.concat(
    [tesseract_df, easyocr_df, paddle_df],
    ignore_index=True
)

comparison = comparison[
    ["OCR Engine", "Language", "CER", "WER", "Character Accuracy"]
]

comparison = comparison.sort_values(
    ["Language", "OCR Engine"]
)

print("=" * 75)
print("BHARATDOC AI - OCR ENGINE COMPARISON")
print("=" * 75)

print(comparison.to_string(index=False))

# Save consolidated results
comparison.to_csv(
    PROJECT_ROOT / "outputs/metrics/ocr_engine_comparison.csv",
    index=False
)

print("\nComparison saved successfully!")


BHARATDOC AI - OCR ENGINE COMPARISON
OCR Engine Language      CER      WER  Character Accuracy
   EasyOCR  english 0.000000 0.000000            1.000000
 PaddleOCR  english 0.000000 0.000000            1.000000
 Tesseract  english 0.000000 0.000000            1.000000
   EasyOCR    hindi 0.413793 0.692308            0.586207
 PaddleOCR    hindi 0.045977 0.153846            0.954023
 Tesseract    hindi 0.057471 0.230769            0.942529
   EasyOCR  kannada 0.069307 0.384615            0.930693
 Tesseract  kannada 0.039604 0.153846            0.960396

Comparison saved successfully!


In [35]:

import json
import re
import unicodedata
import pandas as pd
from jiwer import cer, wer

# --------------------------------------------------
# 1. LOAD ORIGINAL GROUND TRUTH
# --------------------------------------------------

with open(
    PROJECT_ROOT / "data/annotations/phase0_ground_truth.json",
    encoding="utf-8"
) as f:
    ground_truth = json.load(f)


# --------------------------------------------------
# 2. LOAD ORIGINAL OCR OUTPUTS
# --------------------------------------------------

ocr_files = {
    "Tesseract": "phase0_ocr.json",
    "EasyOCR": "easyocr_results.json",
    "PaddleOCR": "paddleocr_results.json"
}

ocr_outputs = {}

for engine, filename in ocr_files.items():

    file_path = PROJECT_ROOT / "data/ocr_outputs" / filename

    with open(file_path, encoding="utf-8") as f:
        ocr_outputs[engine] = json.load(f)

    print(f"Loaded {engine}: {list(ocr_outputs[engine].keys())}")


# --------------------------------------------------
# 3. NORMALIZATION FUNCTIONS
# --------------------------------------------------

def normalize_raw(text):
    """
    Only standardize whitespace.
    Preserve original characters and punctuation.
    """
    return re.sub(r"\s+", " ", text).strip()


def normalize_unicode(text):
    """
    Standardize equivalent numeral and punctuation
    representations without correcting actual words.
    """

    normalized = unicodedata.normalize("NFC", text)

    # Convert Unicode decimal digits to ASCII.
    converted = []

    for char in normalized:
        try:
            converted.append(str(unicodedata.decimal(char)))
        except (ValueError, TypeError):
            converted.append(char)

    normalized = "".join(converted)

    # Standardize colon variants.
    normalized = normalized.replace("ः", ":")
    normalized = normalized.replace("：", ":")

    # Standardize whitespace.
    normalized = re.sub(r"\s+", " ", normalized)

    # Standardize spacing around separators.
    normalized = re.sub(r"\s*([:/-])\s*", r"\1", normalized)

    return normalized.strip()


# --------------------------------------------------
# 4. CALCULATE METRICS FOR ALL ENGINES
# --------------------------------------------------

languages = ["english", "hindi", "kannada"]

all_results = []

for engine, predictions in ocr_outputs.items():

    for language in languages:

        # Do not invent missing language results.
        if language not in predictions:
            print(f"Skipping {engine} - {language}: unavailable")
            continue

        reference = ground_truth[language]["text"]
        prediction = predictions[language]

        # Raw evaluation
        raw_reference = normalize_raw(reference)
        raw_prediction = normalize_raw(prediction)

        raw_cer = cer(raw_reference, raw_prediction)
        raw_wer = wer(raw_reference, raw_prediction)

        # Normalized evaluation
        norm_reference = normalize_unicode(reference)
        norm_prediction = normalize_unicode(prediction)

        normalized_cer = cer(
            norm_reference,
            norm_prediction
        )

        normalized_wer = wer(
            norm_reference,
            norm_prediction
        )

        all_results.append({
            "OCR Engine": engine,
            "Language": language,
            "Raw CER": raw_cer,
            "Normalized CER": normalized_cer,
            "Raw WER": raw_wer,
            "Normalized WER": normalized_wer,
            "Raw Character Accuracy": max(0, 1 - raw_cer),
            "Normalized Character Accuracy": max(
                0, 1 - normalized_cer
            )
        })


# --------------------------------------------------
# 5. CREATE COMPARISON TABLE
# --------------------------------------------------

final_df = pd.DataFrame(all_results)

final_df = final_df.sort_values(
    ["Language", "OCR Engine"]
).reset_index(drop=True)

print("\n" + "=" * 100)
print("BHARATDOC AI - UNIFIED OCR EVALUATION")
print("=" * 100)

print(final_df.to_string(index=False))


# --------------------------------------------------
# 6. SAVE FINAL METRICS
# --------------------------------------------------

output_path = (
    PROJECT_ROOT /
    "outputs/metrics/unified_ocr_evaluation.csv"
)

final_df.to_csv(output_path, index=False)

print("\nUnified evaluation saved:", output_path)


# --------------------------------------------------
# 7. VERIFY RAW METRIC CONSISTENCY
# --------------------------------------------------

print("\nEVALUATION VALIDATION")

assert final_df["Raw CER"].between(0, 1).all()
assert final_df["Raw WER"].between(0, 1).all()

assert (
    final_df["Raw Character Accuracy"]
    == (1 - final_df["Raw CER"]).clip(0, 1)
).all()

assert (
    final_df["Normalized Character Accuracy"]
    == (1 - final_df["Normalized CER"]).clip(0, 1)
).all()

print("All metric validation checks passed!")


Loaded Tesseract: ['english', 'hindi', 'kannada']
Loaded EasyOCR: ['english', 'hindi', 'kannada']
Loaded PaddleOCR: ['english', 'hindi']
Skipping PaddleOCR - kannada: unavailable

BHARATDOC AI - UNIFIED OCR EVALUATION
OCR Engine Language  Raw CER  Normalized CER  Raw WER  Normalized WER  Raw Character Accuracy  Normalized Character Accuracy
   EasyOCR  english 0.000000        0.000000 0.000000           0.000                1.000000                       1.000000
 PaddleOCR  english 0.000000        0.000000 0.000000           0.000                1.000000                       1.000000
 Tesseract  english 0.000000        0.000000 0.000000           0.000                1.000000                       1.000000
   EasyOCR    hindi 0.413793        0.256098 0.692308           0.375                0.586207                       0.743902
 PaddleOCR    hindi 0.045977        0.048780 0.153846           0.250                0.954023                       0.951220
 Tesseract    hindi 0.057471    